In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gpt-oss:20b"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
configuracion = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

# Verificar integridad
assert sha_casos == configuracion["cases_sha256"]

assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["prompt_version"] == "V3"
assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Consultar Ollama
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as f:
    version_ollama = json.load(f)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Modelo no instalado."

# Características reales del modelo
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as f:
    detalles = json.load(f)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Capacidades:", detalles.get("capabilities"))
print("Parámetros:", detalles.get("parameters"))

print("\nArchivos y conexión: correctos")


Python: 3.11.16
Ollama: 0.40.13
Modelo: gpt-oss:20b
Identificador: f38aa0c53da5
Casos verificados: 6
SHA256: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Capacidades: ['completion', 'tools', 'thinking']
Parámetros: temperature                    1

Archivos y conexión: correctos


In [2]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request

ARCHIVO = "resultados.jsonl"
CASO_ID = "DIAG-GPTOSS-20B-01"
CONFIG_MODELO = "gptoss_20b_medium_v1"

# 1. Caso diagnóstico común
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# 2. Configuración compartida V3
solicitud = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud["model"] = MODELO

# gpt-oss utiliza niveles de razonamiento
solicitud["think"] = "medium"

solicitud["options"] = {
    "num_ctx": 4096,
    "num_predict": 3072,
    "temperature": 1
}

# Conservar los prompts V3 originales
solicitud["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id=CASO_ID,
            contenido=contenido
        )
    }
]

# 3. Verificaciones
assert MODELO == "gpt-oss:20b"
assert configuracion["prompt_version"] == "V3"
assert "thinking" in detalles.get("capabilities", [])
assert solicitud["think"] == "medium"
assert sha_casos == configuracion["cases_sha256"]
assert modelo_info["digest"].startswith("f38aa0c53da5")

# Evitar duplicar el diagnóstico
try:
    with open(ARCHIVO, encoding="utf-8") as f:
        anteriores = [
            json.loads(line)
            for line in f if line.strip()
        ]
except FileNotFoundError:
    anteriores = []

assert not any(
    r.get("case_id") == CASO_ID
    and r.get("configuracion_modelo") == CONFIG_MODELO
    for r in anteriores
), "El diagnóstico ya está registrado."

# 4. Preparar registro
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "diagnostico",
    "configuracion_modelo": CONFIG_MODELO,
    "prompt_version": "V3",
    "modelo_digest": modelo_info["digest"],
    "ollama_version": version_ollama,
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 5. Ejecutar diagnóstico local
try:
    peticion = Request(
        configuracion["api_url"],
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente_local.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    respuesta_final = mensaje.get("content", "").strip()
    razonamiento = mensaje.get("thinking", "")

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif respuesta.get("done_reason") == "stop" and respuesta_final:
        registro["estado"] = (
            "ok" if razonamiento else "revisar_modo"
        )

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Guardar la ejecución completa
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 7. Mostrar métricas y respuesta final
print("Caso:", CASO_ID)
print("Configuración:", CONFIG_MODELO)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    for campo in (
        "total_duration",
        "load_duration",
        "prompt_eval_duration",
        "eval_duration"
    ):
        valor = r.get(campo)

        if valor is not None:
            print(campo, round(valor / 1e9, 3), "s")

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro guardado en resultados.jsonl")


Caso: DIAG-GPTOSS-20B-01
Configuración: gptoss_20b_medium_v1
Estado: ok
Tiempo cliente: 47.785 s
Tokens entrada: 414
Tokens generados: 446
Terminación: stop
Razonamiento separado: True
total_duration 47.783 s
load_duration 15.487 s
prompt_eval_duration 12.018 s
eval_duration 20.267 s

RESPUESTA FINAL:

**Evidencia**  
- [E1] Autor X publicó un artículo sobre bases de datos académicas.  
- [E2] Autor Y publicó un artículo sobre recuperación de información.  

**Interpretación**  
Los dos autores tienen trabajos publicados en áreas que son complementarias: el manejo de bases de datos académicas (E1) y la recuperación de información (E2). Esta coincidencia temática sugiere que podrían interesar por temas relacionados, lo que puede ser un factor que el modelo predictor haya considerado para generar una predicción favorable de coautoría.  

**Limitaciones**  
- No se dispone de datos sobre colaboraciones previas entre Autor X y Autor Y.  
- No se conocen las métricas internas del modelo ni 

In [3]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "gpt-oss:20b"
API = "http://127.0.0.1:11434"
ARCHIVO = "resultados.jsonl"
CONFIG_MODELO = "gptoss_20b_medium_v1"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["prompt_version"] == "V3"
assert config["api_url"] == API + "/api/chat"

# 2. Recuperar diagnóstico exitoso
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [
        json.loads(line)
        for line in f if line.strip()
    ]

diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-GPTOSS-20B-01"
    and r["configuracion_modelo"] == CONFIG_MODELO
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["model"] == MODELO
assert diagnostico["solicitud"]["think"] == "medium"

assert diagnostico["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 3072,
    "temperature": 1
}

assert diagnostico["sha256_casos"] == sha_casos
assert diagnostico["sha256_configuracion"] == sha_config

assert diagnostico["solicitud"]["messages"][0]["content"] == (
    config["system_prompt"]
)

# 3. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_info = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_info["digest"] == diagnostico["modelo_digest"]

# Evitar casos duplicados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == CONFIG_MODELO
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado", flush=True)
            continue

        # Recuperar solicitud exacta del diagnóstico
        solicitud = copy.deepcopy(
            diagnostico["solicitud"]
        )

        # Sustituir únicamente el caso y sus evidencias
        solicitud["messages"][1]["content"] = (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Preparar registro
        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": CONFIG_MODELO,
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        print("Ejecutando:", case_id, flush=True)
        inicio = time.perf_counter()

        # 5. Consultar Ollama local
        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=900) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})
            contenido_final = mensaje.get("content", "").strip()
            razonamiento = mensaje.get("thinking", "")

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"

            elif respuesta.get("done_reason") == "stop" and contenido_final:
                registro["estado"] = (
                    "ok" if razonamiento else "revisar_modo"
                )

            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # 6. Guardar inmediatamente cada ejecución
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})
        mensaje = r.get("message", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s",
            "| thinking:", bool(mensaje.get("thinking")),
            flush=True
        )

print("\nArchivo actualizado:", ARCHIVO)


Ejecutando: S-005
S-005 | ok | 660 tokens | 20.475 s | thinking: True
Ejecutando: S-006
S-006 | ok | 645 tokens | 13.853 s | thinking: True
Ejecutando: S-007
S-007 | ok | 770 tokens | 16.901 s | thinking: True
Ejecutando: S-008
S-008 | ok | 639 tokens | 14.566 s | thinking: True
Ejecutando: S-009
S-009 | ok | 564 tokens | 11.887 s | thinking: True
Ejecutando: S-010
S-010 | ok | 652 tokens | 13.562 s | thinking: True

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "gpt-oss:20b"
CASO_ID = "S-009"
CONFIG_HIGH = "gptoss_20b_high_piloto_v1"
API = "http://127.0.0.1:11434"

# 1. Verificar archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)
assert config["prompt_version"] == "V3"

# 2. Recuperar S-009 con razonamiento medium
with open(ARCHIVO, encoding="utf-8") as f:
    registros = [
        json.loads(line)
        for line in f if line.strip()
    ]

original = next(
    r for r in registros
    if r["case_id"] == CASO_ID
    and r["configuracion_modelo"] == "gptoss_20b_medium_v1"
    and r["grupo"] == "comparacion"
)

assert original["estado"] == "ok"
assert original["solicitud"]["model"] == MODELO
assert original["solicitud"]["think"] == "medium"
assert original["sha256_casos"] == sha_casos
assert original["sha256_configuracion"] == sha_config

assert original["solicitud"]["options"] == {
    "num_ctx": 4096,
    "num_predict": 3072,
    "temperature": 1
}

# Evitar duplicar esta prueba
assert not any(
    r.get("case_id") == CASO_ID
    and r.get("configuracion_modelo") == CONFIG_HIGH
    for r in registros
), "La prueba high ya está registrada."

# 3. Cambiar únicamente el nivel de razonamiento
solicitud = copy.deepcopy(original["solicitud"])
solicitud["think"] = "high"

# 4. Verificar identidad del modelo
cliente = build_opener(ProxyHandler({}))

with cliente.open(API + "/api/tags", timeout=10) as f:
    modelos = json.load(f)["models"]

modelo_actual = next(
    m for m in modelos if m["name"] == MODELO
)

assert modelo_actual["digest"] == original["modelo_digest"]

# 5. Preparar registro independiente
registro = {
    "execution_id": str(uuid4()),
    "case_id": CASO_ID,
    "grupo": "exploracion",
    "configuracion_modelo": CONFIG_HIGH,
    "referencia_execution_id": original["execution_id"],
    "motivo": "Evaluar predicción no favorable con razonamiento high",
    "prompt_version": "V3",
    "modelo_digest": original["modelo_digest"],
    "ollama_version": original["ollama_version"],
    "sha256_casos": sha_casos,
    "sha256_configuracion": sha_config,
    "fecha_inicio_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

# 6. Ejecutar prueba high
try:
    peticion = Request(
        API + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=900) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    mensaje = respuesta.get("message", {})
    contenido_final = mensaje.get("content", "").strip()
    razonamiento = mensaje.get("thinking", "")

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"

    elif respuesta.get("done_reason") == "stop" and contenido_final:
        registro["estado"] = (
            "ok" if razonamiento else "revisar_modo"
        )

    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 7. Guardar sin sobrescribir los resultados originales
with open(ARCHIVO, "a", encoding="utf-8") as f:
    f.write(
        json.dumps(registro, ensure_ascii=False) + "\n"
    )

# 8. Mostrar resultados
print("Caso:", CASO_ID)
print("Configuración:", CONFIG_HIGH)
print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]
    mensaje = r.get("message", {})

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        mensaje.get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(mensaje.get("content", ""))

else:
    print("ERROR:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Caso: S-009
Configuración: gptoss_20b_high_piloto_v1
Estado: ok
Tiempo cliente: 39.413 s
Tokens entrada: 431
Tokens generados: 1571
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

**Evidencia**  
- **E1**: Artículo *P-Q01* escrito por Autor Q (2024) sobre “retrieval‑augmented generation para responder preguntas científicas”.  
- **E2**: Artículo *P-R01* escrito por Autor R (2025) sobre “evaluación de métodos de recuperación de evidencia para la respuesta a preguntas científicas”.  

No aparece en la evidencia proporcionada ningún documento que liste a Autor Q y Autor R como coautores, ni se menciona colaboración directa, proyecto compartido o afiliación institucional común.

**Interpretación**  
El hecho de que ambos autores publiquen en temáticas afines (retrieval‑augmented generation y métodos de recuperación de evidencia) no implica ni prueba una colaboración. La ausencia de publicaciones conjuntas, como se evidencia en E1 y E2, indica que, al menos hasta la fecha d